# Wildfire Classification Using Satellite Images

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from tensorflow.keras.models import load_model
from PIL import Image

## Loading Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip "/content/test.zip"

In [ ]:
fire_dir = "wildfire"    # folder w/ wildfire
no_fire_dir = "nowildfire"  # folder w/ normal

In [ ]:

fire_images = [os.path.join(fire_dir, f) for f in os.listdir(fire_dir)]
no_fire_images = [os.path.join(no_fire_dir, f) for f in os.listdir(no_fire_dir)]

#1=wildfire, 0=no wildfire
fire_labels = [1] * len(fire_images)
no_fire_labels = [0] * len(no_fire_images)

all_image_paths = fire_images + no_fire_images
all_labels = fire_labels + no_fire_labels

In [ ]:
#Check for invalid images, error was happening earlier in training process
from PIL import Image, UnidentifiedImageError

cleaned_image_paths = []
cleaned_labels = []


for path, label in zip(all_image_paths, all_labels):
    try:
        with Image.open(path) as img:
            img.load()
        cleaned_image_paths.append(path)
        cleaned_labels.append(label)
    except (IOError, SyntaxError, UnidentifiedImageError) as e:
        print(f"⚠️ Invalid image: {path} — {e}")

print(f"\n Valid images: {len(cleaned_image_paths)} / {len(all_image_paths)}")

# Update variables
all_image_paths = cleaned_image_paths
all_labels = cleaned_labels


In [ ]:
# Display sample images from both classes
plt.figure(figsize=(12, 5))

#Wildfire
plt.subplot(1, 2, 1)
sample_fire = plt.imread(fire_images[0])
plt.imshow(sample_fire)
plt.title(f"Wildfire (1)\nShape: {sample_fire.shape}")
plt.axis('off')

#No Wildfire
plt.subplot(1, 2, 2)
sample_no_fire = plt.imread(no_fire_images[0])
plt.imshow(sample_no_fire)
plt.title(f"No Wildfire (0)\nShape: {sample_no_fire.shape}")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_image_paths,
    all_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_labels
)

## Preprocessing
*   Reading, Decoding, Sizing, Scaling



In [ ]:
def load_and_preprocess(image_path, label):
    try:
      img = tf.io.read_file(image_path)
      img = tf.image.decode_jpeg(img, channels=3)
      img = tf.image.resize(img, [128, 128]) / 255.0
      return img, tf.cast(label, tf.float32)

    #additional error handling
    except Exception:
        return tf.zeros((128, 128, 3)), tf.constant(0.0, dtype=tf.float32)

output_signature = (
    tf.TensorSpec(shape=(128, 128, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32)
)


train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(
    load_and_preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
).batch(32).cache().prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(
    load_and_preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
).batch(32).cache().prefetch(tf.data.AUTOTUNE)


print("Train dataset spec:", train_ds.element_spec)

In [ ]:

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i+1)
        plt.imshow(images[i].numpy())
        plt.title(f"Label: {labels[i].numpy()}")
        plt.axis("off")
plt.suptitle("Augmented Training Samples (Post-Resizing/Normalization)", y=1.05)
plt.show()

In [ ]:

class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = {0: class_weights[0], 1: class_weights[1]}

print("Class weights:", class_weights)


## Building Model

In [ ]:
# data_augmentation = tf.keras.Sequential([
#     layers.RandomFlip("horizontal_and_vertical"),
#     layers.RandomRotation(0.2),
#     layers.RandomZoom(0.1),
#     layers.RandomBrightness(0.1),
# ])

In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(128, 128, 3))

    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)


model = build_model()
model.summary()

## Train Model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [ ]:
# Check one batch
sample_images, sample_labels = next(iter(train_ds))
print("Image batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)
print("Sample labels:", sample_labels[:5].numpy())

# Visualize samples of satellite data before I train
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(sample_images[i])
    plt.title(f"Label: {sample_labels[i].numpy()}")
    plt.axis('off')
plt.show()

In [ ]:
# Check label shape
for images, labels in train_ds.take(1):
    print("Label shape:", labels.shape)
    print("Sample labels:", labels[:5].numpy())

In [ ]:
print(f"Train dataset element spec: {train_ds.element_spec}")
print(f"Validation dataset element spec: {val_ds.element_spec}")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2
    ),

    # tf.keras.callbacks.ModelCheckpoint(
    #     "resume_training.h5",
    #     save_best_only=False,
    #     save_weights_only=False,
    #     monitor="val_loss",
    #     mode="min"
    # )
]


history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    callbacks=callbacks
)

In [ ]:
# model = load_model('resume_training.h5')

# # Continue training
# history = model.fit(
#     train_ds,
#     epochs=30,
#     initial_epoch=0,  # Start from epoch 11
#     validation_data=val_ds
# )


## Evaluation Metrics

In [ ]:
# Training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()

#Loss Function
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

# Evaluate on validation set
val_results = model.evaluate(val_ds)
print(f"Validation Accuracy: {val_results[1]:.2%}")
print(f"Validation Precision: {val_results[2]:.2%}")
print(f"Validation Recall: {val_results[3]:.2%}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(val_ds)
y_pred_labels = np.argmax(y_pred, axis=1)
true_labels = val_labels  # or extract from val_ds if using tf.data

cm = confusion_matrix(true_labels, y_pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Wildfire", "Wildfire"])
disp.plot(cmap='Blues')
plt.show()


## Final Test

In [ ]:
model.save('best_model.keras')